# 1. Configuración del entorno

In [ ]:
# Mount Google Drive for Colab
from google.colab import drive
drive.mount('/content/drive')

# Librerias utilizadas
import requests
import os
import zipfile
import io

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 2. Scraping de los módulos ENAHO (2020–2024)

## 2.1. Configuración

In [ ]:
RUTA_BASE = '/content/drive/MyDrive/Colab Notebooks/Curso 1/Data/IN/'

# Mapeo de IDs por año identificados inspeccionando el portal del INEI
config_años = {
    "2020": "737",
    "2021": "759",
    "2022": "784",
    "2023": "906",
    "2024": "966"
}

# Módulos
modulos = ["02", "03", "04", "34"]
base_url = "https://proyectos.inei.gob.pe/iinei/srienaho/descarga/CSV/"

print(f"Iniciando descarga en: {RUTA_BASE}\n")

Iniciando descarga en: /content/drive/MyDrive/Colab Notebooks/Curso 1/Data/IN/



## 2.2. Bucle de descarga

In [ ]:
for anio, id_anio in config_años.items():
    # Crear carpeta por año
    ruta_anio = os.path.join(RUTA_BASE, anio)
    os.makedirs(ruta_anio, exist_ok=True)

    print(f"--- PROCESANDO AÑO {anio} (ID: {id_anio}) ---")

    for mod in modulos:
        url = f"{base_url}{id_anio}-Modulo{mod}.zip"

        try:
            r = requests.get(url, stream=True, timeout=30)

            if r.status_code == 200:
                print(f"Descargando Módulo {mod}...")

                # Descomprimir directamente para guardar solo los CSV
                with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                    for filename in z.namelist():
                        if filename.upper().endswith('.CSV'):
                            # Extraer en la carpeta del año
                            z.extract(filename, path=ruta_anio)
                            print(f"   ✓ Extraído: {filename}")
            else:
                print(f"   ✗ No encontrado: Módulo {mod} (URL: {url})")

        except Exception as e:
            print(f"   ! Error en Módulo {mod}: {e}")
    print("\n")

print("="*30)
print("DESCARGA COMPLETA Y ORGANIZADA")
print("="*30)

--- PROCESANDO AÑO 2020 (ID: 737) ---
Descargando Módulo 02...
   ✓ Extraído: 737-Modulo02/ENAHO-TABLA-CIUO-88.csv
   ✓ Extraído: 737-Modulo02/ENAHO-TABLA-CNO-2015.csv
   ✓ Extraído: 737-Modulo02/Enaho01-2020-200.csv
Descargando Módulo 03...
   ✓ Extraído: 737-Modulo03/Enaho01A-2020-300.csv
   ✓ Extraído: 737-Modulo03/Enaho01A-2020-300A.csv
Descargando Módulo 04...
   ✓ Extraído: 737-Modulo04/Enaho01a-2020-400.csv
Descargando Módulo 34...
   ✓ Extraído: 737-Modulo34/Sumaria-2020-12g.csv
   ✓ Extraído: 737-Modulo34/Sumaria-2020.csv


--- PROCESANDO AÑO 2021 (ID: 759) ---
Descargando Módulo 02...
   ✓ Extraído: 759-Modulo02/Enaho01-2021-200.csv
Descargando Módulo 03...
   ✓ Extraído: 759-Modulo03/Enaho01a-2021-300.csv
   ✓ Extraído: 759-Modulo03/Enaho01A-2021-300A.csv
Descargando Módulo 04...
   ✓ Extraído: 759-Modulo04/Enaho01a-2021-400.csv
Descargando Módulo 34...
   ✓ Extraído: 759-Modulo34/Sumaria-2021-12g.csv
   ✓ Extraído: 759-Modulo34/Sumaria-2021.csv


--- PROCESANDO AÑO 2022 (ID

# 3. Scraping del archivo IDH

## 3.1. Localización del enlace

In [ ]:
# La URL fue identificada inspeccionando el atributo href del botón de descarga del Anexo 1 en el portal del PNUD Perú:
# clase: "text-link arrow-3 download-btn flex-container"

RUTA_IDH  = '/content/drive/MyDrive/Colab Notebooks/Curso 1/Data/IN/IDH/'
os.makedirs(RUTA_IDH, exist_ok=True)

URL_IDH = 'https://www.undp.org/sites/g/files/zskgke326/files/2025-06/anexo_1-idh_2017-2024_a_nivel_distrital.xlsx'

## 3.2. Descarga y almacenamiento

In [ ]:
# Descarga y almacenamiento con headers de navegador
try:
    headers = {
        # Simulamos que la petición proviene de un navegador Chrome real (Windows).
        # Sin esto, el servidor identifica la petición como un script automático y devuelve un error 403 (Forbidden).
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        # Indicamos al servidor desde qué página se originó la descarga.
        # El servidor del PNUD verifica que la petición provenga de su propio portal antes de permitir la descarga del archivo.
        'Referer': 'https://www.undp.org/es/peru/publicaciones/informe-sobre-desarrollo-humano-2025-actuar-confiar-y-conectar-caminos',
        # Finalmente, especificamos el tipo de archivo que se espera recibir.
        # En este caso, el formato .xlsx usado por el Anexo 1 del IDH.
        'Accept': 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet,*/*',
    }

    r = requests.get(URL_IDH, headers=headers, timeout=60)

    if r.status_code == 200:
        ruta_archivo = os.path.join(RUTA_IDH, 'IDH_2017_2024_distrital.xlsx')
        with open(ruta_archivo, 'wb') as f:
            f.write(r.content)
        print(f"✓ Archivo IDH guardado en: {ruta_archivo}")
    else:
        print(f"✗ Error al descargar: status {r.status_code}")

except Exception as e:
    print(f"! Error: {e}")

print("="*30)
print("DESCARGA COMPLETA Y ORGANIZADA")
print("="*30)

✓ Archivo IDH guardado en: /content/drive/MyDrive/Colab Notebooks/Curso 1/Data/IN/IDH/IDH_2017_2024_distrital.xlsx
DESCARGA COMPLETA Y ORGANIZADA


# 4. Verificación de archivos descargados

## 4.1. Verificación de módulos ENAHO (2020-2024)

In [ ]:
import pandas as pd
import os

# Verificar que los archivos existen y mostrar dimensiones
modulos_principales = {
    "Módulo 200": ("200", "Enaho01"),
    "Módulo 300": ("300", "Enaho01"),
    "Módulo 400": ("400", "Enaho01"),
    "Módulo 34":  ("34",  "Sumaria")
}

for anio in config_años.keys():
    print(f"\n--- AÑO {anio} ---")
    ruta_anio = os.path.join(RUTA_BASE, anio)

    for carpeta in os.listdir(ruta_anio):
        ruta_carpeta = os.path.join(ruta_anio, carpeta)
        for archivo in os.listdir(ruta_carpeta):
            if archivo.endswith('.csv') and not any(
                x in archivo for x in ['TABLA', 'CIUO', 'CNO', 'CIIU', 'PAISES', 'UBIGEO']
            ):
                ruta_csv = os.path.join(ruta_carpeta, archivo)
                df = pd.read_csv(ruta_csv, nrows=5, low_memory=False, encoding='latin-1')
                print(f"  ✓ {archivo}: {df.shape[1]} columnas")


--- AÑO 2020 ---
  ✓ Enaho01-2020-200.csv: 43 columnas
  ✓ Enaho01A-2020-300.csv: 568 columnas
  ✓ Enaho01A-2020-300A.csv: 30 columnas
  ✓ Enaho01a-2020-400.csv: 1052 columnas
  ✓ Sumaria-2020-12g.csv: 252 columnas
  ✓ Sumaria-2020.csv: 179 columnas

--- AÑO 2021 ---
  ✓ Enaho01-2021-200.csv: 41 columnas
  ✓ Enaho01a-2021-300.csv: 513 columnas
  ✓ Enaho01A-2021-300A.csv: 28 columnas
  ✓ Enaho01a-2021-400.csv: 984 columnas
  ✓ Sumaria-2021-12g.csv: 266 columnas
  ✓ Sumaria-2021.csv: 187 columnas

--- AÑO 2022 ---
  ✓ Enaho01-2022-200.csv: 40 columnas
  ✓ Enaho01a-2022-300.csv: 511 columnas
  ✓ Enaho01a-2022-400.csv: 977 columnas
  ✓ Sumaria-2022-12g.csv: 246 columnas
  ✓ Sumaria-2022.csv: 167 columnas

--- AÑO 2023 ---
  ✓ Enaho01-2023-200.csv: 40 columnas
  ✓ Enaho01A-2023-300.csv: 511 columnas
  ✓ Enaho01A-2023-400.csv: 940 columnas
  ✓ Sumaria-2023-12g.csv: 242 columnas
  ✓ Sumaria-2023.csv: 163 columnas

--- AÑO 2024 ---
  ✓ Enaho01-2024-200.csv: 40 columnas
  ✓ Enaho01A-2024-300.c

## 4.2. Verificación de IDH

In [ ]:
df_idh = pd.read_excel(
    os.path.join(RUTA_IDH, 'IDH_2017_2024_distrital.xlsx'),
    nrows=5
)

print(f"✓ IDH cargado correctamente")
print(f"  Dimensiones (muestra): {df_idh.shape}")
print(f"  Columnas: {df_idh.columns.tolist()}")
df_idh.head()

✓ IDH cargado correctamente
  Dimensiones (muestra): (5, 58)
  Columnas: ['Unnamed: 0', 'Unnamed: 1', 2017, 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 2018, 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 2019, 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 2020, 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 2021, 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 2022, 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 2023, 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 50', 2024, 'Unnamed: 52', 'Unnamed: 53', 'Unnamed: 54', 'Unnamed: 55', 'Unnamed: 56', 'Unnamed: 57']


,Unnamed: 0,Unnamed: 1,2017,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,2018,...,Unnamed: 48,Unnamed: 49,Unnamed: 50,2024,Unnamed: 52,Unnamed: 53,Unnamed: 54,Unnamed: 55,Unnamed: 56,Unnamed: 57
0,NaN,NaN,Población,Esperanza de Vida al Nacer,Valor esperado de años de educación en el hogar,Años de educación,Ingreso real per cápita del hogar,IDH,Ranking,Población,...,Ingreso real per cápita del hogar,IDH,Ranking,Población,Esperanza de Vida al Nacer,Valor esperado de años de educación en el hogar,Años de educación,Ingreso real per cápita del hogar,IDH,Ranking
1,Ubigeo,DEPARTAMENTO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,PROVINCIA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,DISTRITO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,000000,PERÚ,31237384,76.619614,0.6528,8.986952,952.043884,0.648446,NaN,31562130,...,892.095093,0.659646,NaN,34038456,78.328949,0.668455,9.452877,896.541748,0.662126,NaN
